In [0]:
from pyspark.sql import functions as F
from delta.tables import DeltaTable

In [0]:
%run /Workspace/Users/maximgarner54@gmail.com/SportsCompanyProject/1_setup/utilities

In [0]:
dbutils.widgets.text("catalog", "SportsProject", "Catalog")
dbutils.widgets.text("data_source", "Customers", "Data Source")

catalog = dbutils.widgets.get("catalog")
data_source = dbutils.widgets.get("data_source")

print(catalog, data_source)

In [0]:
base_path = f's3://sportscompanyproject-maximgarner/full_load/{data_source}'
df = (
spark.read.format("csv")
    .option("header", "true")
    .option("inferSchema", "true")
    .load(base_path)
    .withColumn("read_timestamp", F.current_timestamp())
    .select("*", "_metadata.file_name", "_metadata.file_size")
)
display(df.limit(10))

In [0]:
df.write\
    .format("delta") \
    .option("delta.enableChangeDataFeed", "true") \
    .mode("overwrite")\
    .saveAsTable(f"{catalog}.{bronze_schema}.{data_source}")

### Silver Processing

In [0]:
df_bronze = spark.sql(f"SELECT * FROM {catalog}.{bronze_schema}.{data_source};")
df_bronze.show(10)

In [0]:
df_duplicates = df_bronze.groupBy("customer_id").count().filter("count > 1")

df_bronze.createOrReplaceTempView("df_bronze")
df_duplicates.createOrReplaceTempView("df_duplicates")
df_duplicateinformation = spark.sql(f"SELECT * FROM df_bronze where customer_id in (SELECT customer_id FROM df_duplicates) order by customer_id;")
display(df_bronze)


In [0]:
df_silver = df_bronze.dropDuplicates(['customer_id'])
display(df_silver)

## Getting rid of spaces around customer names

In [0]:

df_silver = df_silver.withColumn("customer_name", F.trim(F.col('customer_name')))
display(df_silver)
#CheckAgainAfter = df_silver.select("*").filter(F.col("customer_name") != F.trim(F.col("customer_name")))
#display(CheckAgainAfter)
#empty as expected

#Checking Distinct Cities

In [0]:
#df_silver.select("city").distinct().show()
#Cities frequently misspelled
#Create a mapping for the city names and restict to allowed cities, otherwise null

city_mapping = {
    'Bengalore' : 'Bengaluru',
    'Bengaluruu' : 'Bengaluru',
    
    'Hyderabadd' : 'Hyderabad',
    'Hyderbad' : 'Hyderabad',

    'NewDelhee' : 'New Delhi',
    'NewDheli' : 'New Delhi',
    'NewDelhi' : 'New Delhi'
}
allowed = ['Bengaluru', 'Hyderabad', 'New Delhi']
df_silver = (df_silver
    .replace(city_mapping, subset = ["city"])
    .withColumn("city",
        F.when(F.col("city").isin(allowed), F.col("city"))
        .when(F.col("city").isNull(), None)
        .otherwise(None))
)
display(df_silver)
df_silver.select("city").distinct().show()

#Now to fix Customer Name column

In [0]:
#df_silver.select("customer_name").distinct().show()

df_silver = df_silver.withColumn("customer_name", 
                F.when(F.col("customer_name").isNull(), None)
                .otherwise(F.initcap(F.col("customer_name")))
)
df_silver.select("customer_name").distinct().orderBy("customer_name").show(truncate=False)

#Check now the cities that have null values

In [0]:
import numpy as np
#df_silver.select("*").filter(F.col("city").isNull()).show()
#Then check where the same customers are based
nullcities = df_silver.select("customer_name").filter(F.col("city").isNull()).toPandas()["customer_name"].tolist()


fullnullcities = df_silver.select("*").filter(F.col("customer_name").isin(nullcities)).show()

#comparative check with spelling out the list
nullcityarray = ["Sprintx Nutrition",
                 "Zenathlete Foods",
                 "Primefuel Nutrition",
                 "Recovery Lane"]

#fullnullcities2 = df_silver.select("*").filter(F.col("customer_name").isin(nullcityarray)).show()

In [0]:
#Create a dataframe with the fixed values to join and coalesce with the null values
df_cityfix_index = {
    789403 : "New Delhi",
    789420 : "Bengaluru",
    789521 : "Hyderabad",
    789603 : "Hyderabad"
} 
df_cityfixer = spark.createDataFrame(
    [(x, y) for x, y in df_cityfix_index.items()],
     ["customer_id", "fixed_city"]
)


df_silver = (
    df_silver.join(df_cityfixer, "customer_id", "left")
        .withColumn("city", F.coalesce("city", "fixed_city"))
        .drop("fixed_city")
)
df_silver.select("city").distinct().show()

#Cast the customer_id to a string value and generally adjust the data so the columns match the gold layer labels of the parent company. Note that the gold layer only has Market = India, so we will merge the customer name and city to make them directly identifable in a broader india market

In [0]:
df_silver = df_silver.withColumn("customer_id", F.col("customer_id").cast("string"))
#df_silver.printSchema()
df_silver = (df_silver
             #Set static values to match parent company
          .withColumn("market", F.lit("India")) 
          .withColumn("platform", F.lit("Sports Bar"))
          .withColumn("channel", F.lit("Acquisitions"))
            #Replace Customer_name and city with single Customer column, unknown if city null
          .withColumn("customer", 
            F.concat_ws("-", "customer_name", F.coalesce(F.col("city"), 
                    F.lit("unknown")))
                    )
)
df_silver.show()

In [0]:
#Write dataframe to silver table

df_silver.write \
    .mode("overwrite") \
    .format("delta") \
    .option("mergeSchema", True) \
    .option("delta.enableChangeDataFeed", True) \
    .saveAsTable(f"{catalog}.{silver_schema}.{data_source}")
    

#Gold data processing

In [0]:
df_gold = df_silver.select("customer_id", "customer_name", "city", "customer", "market", "platform", "channel")
display(df_gold)

In [0]:
#Write dataframe to gold table

df_gold.write \
    .mode("overwrite") \
    .format("delta") \
    .option("delta.enableChangeDataFeed", True) \
    .saveAsTable(f"{catalog}.{gold_schema}.sb_dim_{data_source}")

In [0]:
#Merge the cleaned child company data with the parent companies

Delta_table = DeltaTable.forName(spark, "sportsproject.gold.dim_customers")
df_childcompany = spark.table("sportsproject.gold.sb_dim_customers").select(
            F.col("customer_id").alias("customer_code"), 
            "customer",
            "market",
            "platform",
            "channel"
)

Delta_table.alias("target").merge(
    source=df_childcompany.alias("source"),
    condition="target.customer_code = source.customer_code"
).whenMatchedUpdateAll().whenNotMatchedInsertAll().execute()